In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import xarray as xr
from parflow.tools.hydrology import calculate_overland_flow_grid

try:
    _paper_figures = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    _paper_figures = _cwd.parent if _cwd.name == "spinup" else _cwd

sys.path.insert(0, str(_paper_figures))
import utils

HOURS_PER_YEAR = utils.ONE_YEAR
HOURS_PER_WEEK = 7 * 24  # hourly timestep → one sample per week
DX = DY = 1000.0
OUTLET_X = utils.OUTLET_X
OUTLET_Y = utils.OUTLET_Y

In [2]:
DOMAIN_ROOT = Path(
    "/glade/derecho/scratch/bwest/drought-ensemble/domains/potomac_without_flow_barrier"
)
RUN_HASH = "fc2a5b1d5cf4b8279c7d6a98de84df89f0bc21961f7fd18bac11228fec55ac9c"

# Mannings scaling experiments (folder suffix encodes factor)
RUN_DIRS = {
    "n = 0.1": DOMAIN_ROOT / "raw_runs_1" / RUN_HASH,
    "n = 0.01": DOMAIN_ROOT / "raw_runs_01" / RUN_HASH,
    "n = 0.001": DOMAIN_ROOT / "raw_runs_old" / RUN_HASH,
}


def _overland_flow_at_outlet(pressure, slopex, slopey, mannings, mask):
    flow = calculate_overland_flow_grid(
        pressure, slopex, slopey, mannings, DX, DY, mask=mask
    )
    return flow[OUTLET_Y, OUTLET_X]


def load_spinup_run(run_dir: Path) -> xr.Dataset:
    """Weekly outlet flow from yearly NetCDF outputs (hourly native timestep)."""
    run_dir = Path(run_dir)
    static_path = run_dir / "run.out.00000.nc"
    year_files = sorted(
        p
        for p in run_dir.glob("run.out.*.nc")
        if p.name != "run.out.00000.nc" and "CLM" not in p.name
    )
    if not year_files:
        raise FileNotFoundError(f"No yearly run.out.*.nc files in {run_dir}")

    static = xr.open_dataset(static_path)
    mask = static.mask.isel(time=0)
    mannings = static.mannings.isel(time=0)
    slopex = static.slopex.isel(time=0)
    slopey = static.slopey.isel(time=0)
    static.close()

    weekly_time = slice(None, None, HOURS_PER_WEEK)
    outlet_chunks = []
    hour_offset = 0

    for path in year_files:
        with xr.open_dataset(path) as year_ds:
            n_hours = year_ds.sizes["time"]

            if "overland_flow" in year_ds.variables:
                # isel time (and outlet) before any compute; only weekly timesteps read
                outlet = (
                    year_ds["overland_flow"]
                    .isel(time=weekly_time, x=OUTLET_X, y=OUTLET_Y)
                    .load()
                )
            else:
                # isel time before overland_flow calculation
                pressure_weekly = year_ds["pressure"].isel(time=weekly_time)
                outlet = xr.apply_ufunc(
                    _overland_flow_at_outlet,
                    pressure_weekly,
                    slopex,
                    slopey,
                    mannings,
                    mask,
                    input_core_dims=[
                        ["z", "y", "x"],
                        ["y", "x"],
                        ["y", "x"],
                        ["y", "x"],
                        ["z", "y", "x"],
                    ],
                    output_core_dims=[[]],
                    vectorize=True,
                ).load()

            n_weeks = outlet.sizes["time"]
            outlet = outlet.assign_coords(
                time=np.arange(
                    hour_offset,
                    hour_offset + n_weeks * HOURS_PER_WEEK,
                    HOURS_PER_WEEK,
                    dtype=np.float64,
                )
            )
            hour_offset += n_hours
            outlet_chunks.append(outlet)

    outlet_flow = xr.concat(outlet_chunks, dim="time")
    data = xr.Dataset({"outlet_flow": outlet_flow})
    data = data.assign_coords(year=data.time / HOURS_PER_YEAR)
    data.attrs["run_dir"] = str(run_dir)
    data.attrs["n_years"] = float(data.time[-1] / HOURS_PER_YEAR)
    data.attrs["sample_interval_hours"] = HOURS_PER_WEEK
    data = data.isel(time=slice(10, None))
    return data


runs = {label: load_spinup_run(path) for label, path in RUN_DIRS.items()}
for label, ds in runs.items():
    print(
        f"{label}: {ds.sizes['time']:,} weekly samples "
        f"({ds.attrs['n_years']:.2f} years)"
    )

/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'cfradial1' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'furuno' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'gamic' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


ERROR 1: PROJ: proj_create_from_database: Open of /glade/work/bwest/conda-envs/droughts/share/proj failed
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'iris' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'odim' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'rainbow' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


n = 0.1: 43 weekly samples (1.00 years)
n = 0.01: 43 weekly samples (1.00 years)
n = 0.001: 43 weekly samples (1.00 years)


In [7]:
fig = go.Figure()
for label, ds in runs.items():
    fig.add_trace(
        go.Scatter(
            x=ds.year,
            y=ds.outlet_flow,
            mode="lines",
            name=label,
        )
    )

fig.update_layout(
    title="Post spinup outlet overland flow (weekly samples)",
    xaxis_title="Simulation time (years)",
    yaxis_title="Outlet overland flow (m³/s)",
    legend_title="Flow barrier factor",
    template="plotly_white",
)
fig.show()

In [4]:
baseline_run =

SyntaxError: invalid syntax (3399849087.py, line 1)